[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/b_31_gpt2_block_pure_solution.ipynb)

# 🔴 Solution: GPT-2 Block without Flax

*Attention & Transformers · Hard*

Reference implementation. Try it yourself in `b_31_gpt2_block_pure.ipynb` first.

---
Problem 13's GPT-2 block with no Flax — attention, MLP and the residual wiring.

### Signature
```python
class CausalSelfAttention:
    def __init__(self, d_model, num_heads, *, key): ...
    def __call__(self, x): ...

class MLP:
    def __init__(self, d_model, *, key): ...
    def __call__(self, x): ...

class GPT2Block:
    def __init__(self, d_model, num_heads, *, key): ...
    def __call__(self, x): ...        # (B, T, d_model) -> (B, T, d_model)
```

`GPT2Block` holds `self.ln1`, `self.attn`, `self.ln2`, `self.mlp` — the same
attribute names as problem 13. `Linear` and `LayerNorm` are given to you.

### Attention: one fused QKV projection
GPT-2 projects all three at once, then splits:

```python
self.qkv = Linear(d_model, 3 * d_model, key=...)
q, k, v = jnp.split(self.qkv(x), 3, axis=-1)
```

One matmul instead of three. Mask the **logits** with `-inf` before the
softmax, not the weights after it.

### MLP: 4x wide, gelu
`Linear(d_model, 4*d_model)` → `gelu` → `Linear(4*d_model, d_model)`. The
`4x` is GPT-2's convention and is where most of the parameters live.

### Pre-norm is the part people get wrong
```python
x = x + self.attn(self.ln1(x))     # norm INSIDE the branch
x = x + self.mlp(self.ln2(x))
```

Not `x = self.ln1(x + self.attn(x))`. The residual stream must stay unnormalised
end to end — that clean path is what lets gradients reach the bottom of a deep
stack. Post-norm (the original 2017 transformer) needs a warmup schedule to
train at all; pre-norm is why modern models do not.

Both forms produce the right shape and both run, so only a numerical
comparison catches a swap.

### Why this exists alongside problem 13
Interview sandboxes ship `jax` but not `flax`. Same class names, same
attribute names — only `rngs=nnx.Rngs(params=0)` becomes
`key=jax.random.key(0)`.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


class Linear:
    """Given to you, as nnx.Linear is in problem 13."""

    def __init__(self, d_in, d_out, *, key):
        self.kernel = jax.random.normal(key, (d_in, d_out)) / jnp.sqrt(d_in)
        self.bias = jnp.zeros((d_out,))

    def __call__(self, x):
        return x @ self.kernel + self.bias


class LayerNorm:
    """Given to you, as nnx.LayerNorm is in problem 13."""

    def __init__(self, d_model, eps=1e-6):
        self.scale = jnp.ones((d_model,))
        self.bias = jnp.zeros((d_model,))
        self.eps = eps

    def __call__(self, x):
        mu = jnp.mean(x, axis=-1, keepdims=True)
        var = jnp.var(x, axis=-1, keepdims=True)
        return (x - mu) / jnp.sqrt(var + self.eps) * self.scale + self.bias


class CausalSelfAttention:
    def __init__(self, d_model, num_heads, *, key):
        assert d_model % num_heads == 0
        self.h = num_heads
        self.d_head = d_model // num_heads
        k1, k2 = jax.random.split(key, 2)
        # One fused projection, GPT-2 style: one matmul instead of three.
        self.qkv = Linear(d_model, 3 * d_model, key=k1)
        self.out = Linear(d_model, d_model, key=k2)

    def __call__(self, x):
        q, k, v = jnp.split(self.qkv(x), 3, axis=-1)
        split = lambda t: t.reshape(*t.shape[:-1], self.h, self.d_head).swapaxes(-3, -2)
        q, k, v = split(q), split(k), split(v)

        s = jnp.einsum("...hqd,...hkd->...hqk", q, k) / jnp.sqrt(
            jnp.asarray(self.d_head, x.dtype)
        )
        T = s.shape[-1]
        # Mask the LOGITS, before the softmax.
        s = jnp.where(jnp.tril(jnp.ones((T, T), dtype=bool)), s, -jnp.inf)

        o = jnp.einsum("...hqk,...hkd->...hqd", jax.nn.softmax(s, axis=-1), v)
        o = o.swapaxes(-3, -2)
        return self.out(o.reshape(*o.shape[:-2], self.h * self.d_head))


class MLP:
    def __init__(self, d_model, *, key):
        k1, k2 = jax.random.split(key, 2)
        self.fc = Linear(d_model, 4 * d_model, key=k1)
        self.proj = Linear(4 * d_model, d_model, key=k2)

    def __call__(self, x):
        return self.proj(jax.nn.gelu(self.fc(x)))


class GPT2Block:
    def __init__(self, d_model, num_heads, *, key):
        ka, km = jax.random.split(key, 2)
        self.ln1 = LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, num_heads, key=ka)
        self.ln2 = LayerNorm(d_model)
        self.mlp = MLP(d_model, key=km)

    def __call__(self, x):
        # Pre-norm: the norm sits INSIDE each branch, never on the residual.
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

block = GPT2Block(8, 2, key=jax.random.key(0))
x = jax.random.normal(jax.random.key(1), (2, 5, 8))
print("out:", block(x).shape)

print("qkv is fused:", block.attn.qkv.kernel.shape, "= (d_model, 3*d_model)")
print("mlp is 4x:   ", block.mlp.fc.kernel.shape)

# Causality: perturbing the last token must not move the first.
alt = x.at[:, 4].add(50.0)
print("\ncausal?", bool(jnp.allclose(block(alt)[:, 0], block(x)[:, 0], atol=1e-4)))

# Pre-norm keeps the residual stream unnormalised.
print("residual present?", not bool(jnp.allclose(block(x), block(x) - x, atol=1e-4)))

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("gpt2_block_pure")